# ISYS2407 Information Systems Solutions & Design  
# Assignment 3: Hyperparameter Tuning — All Features  

#### Student Name: Lam Le
#### Student Number: s4032582  
#### Dataset: All Features (PKL files)  

---

### Purpose  
This notebook performs **Hyperparameter Tuning** on two classification models — **K-Nearest Neighbours (Model 1)** and **Random Forest (Model 2)** — using the *All Features* dataset.  
The objective is to find the best parameter combinations that maximise **macro F1-score**, with particular attention to improving performance for the **minority class**.  

The tuning process will:  
- Load the preprocessed training and testing datasets (`X_train`, `y_train`, `X_test`, `y_test`) from PKL files.  
- Use **GridSearchCV** with **5-fold cross-validation** to explore hyperparameter spaces.  
- Evaluate each model using **macro F1-score** for balanced performance across classes.  
- Identify and report the **best-performing hyperparameters** for both models.  

---

### Outcome  
- Tuned **KNN** and **Random Forest** classifiers using the *All Features* dataset.  
- Determined **optimal hyperparameters** that improve balance and generalisation.  
- Reported **mean±std cross-validation F1-scores** for model comparison.  
- Saved final **tuned models** and **GridSearchCV results** for reproducibility.


## 1 Import libraries 

In [4]:
# Library for pickling
import joblib

# Miscellabeous libraries
import numpy as np
import pandas as pd
#import collections
#import time

# Model libraries
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import GaussianNB

# Library for performing grid search
from sklearn.model_selection import GridSearchCV

# Metrics library
#from sklearn.metrics import accuracy_score
#from sklearn.metrics import confusion_matrix
#from sklearn.metrics import recall_score
#from sklearn.metrics import precision_score
from sklearn.metrics import f1_score
#from sklearn.metrics import classification_report
#from sklearn.metrics import roc_curve
#from sklearn.metrics import auc

### 2 Model 1: KNN_All Features

### 2.1 Load pkl file

In [5]:
# 2.1 Load PKL File (Model 1 - KNN_All Features)

import joblib

# Load the pickled train and test data
X_train = joblib.load("X_train_1.pkl")
y_train = joblib.load("y_train_1.pkl")
X_test = joblib.load("X_test_1.pkl")
y_test = joblib.load("y_test_1.pkl")

# Check dataset shapes
print("Train data:", X_train.shape, y_train.shape)
print("Test data :", X_test.shape, y_test.shape)

# Preview first few rows
display(X_train.head())


Train data: (4800, 11) (4800,)
Test data : (1200, 11) (1200,)


,age,yrs_experience,family_size,education_level,income,mortgage_amt,credit_card_acct,credit_card_spend,share_trading_acct,fixed_deposit_acct,online_acct
1207,59.0,3.0,4.0,2,111.0,0,0,0.0,0,0,1
1429,49.0,2.0,4.0,3,74.0,220,1,2.5,0,0,1
1347,43.0,1.0,4.0,2,152.0,0,0,0.0,0,0,0
1801,42.0,35.0,1.0,1,87.0,76,1,3.2,0,0,1
4870,57.0,41.0,4.0,3,95.0,105,0,0.0,0,0,0


### 2.2 Optimising Hyperparameters

In [6]:
# 2.2 Optimising Hyperparameters (Model 1 — KNN_All Features)

from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV, StratifiedKFold

# Pipeline: scale -> KNN
knn_pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", KNeighborsClassifier())
])

# Search space (coarse → fine ready)
param_grid = [
    {"clf__n_neighbors": range(1, 76, 5)},
    {"clf__n_neighbors": range(1, 76, 5),
     "clf__weights": ["uniform", "distance"]},
    {"clf__n_neighbors": range(1, 76, 5),
     "clf__weights": ["uniform", "distance"],
     "clf__p": [1, 2],
     "clf__algorithm": ["auto", "kd_tree", "ball_tree"],
     "clf__leaf_size": [15, 30, 45]}
]

# 5-fold stratified CV; scoring = F1 for positive class
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
grid = GridSearchCV(
    estimator=knn_pipe,
    param_grid=param_grid,
    scoring="f1",
    cv=cv,
    n_jobs=-1,
    verbose=0
)

grid.fit(X_train, y_train)

print(f"Best params: {grid.best_params_}, score: {grid.best_score_:0.2f}")

# keep the tuned model for later steps
knn_best = grid.best_estimator_


Best params: {'clf__n_neighbors': 6, 'clf__weights': 'distance'}, score: 0.64


### 3 Model 2: Random_Forest_All Features

### 3.1 Load pkl file

In [7]:
# 2.1 Load PKL File (Model 1 - KNN_All Features)

import joblib

# Load the pickled train and test data
X_train = joblib.load("X_train_2.pkl")
y_train = joblib.load("y_train_2.pkl")
X_test = joblib.load("X_test_2.pkl")
y_test = joblib.load("y_test_2.pkl")

# Check dataset shapes
print("Train data:", X_train.shape, y_train.shape)
print("Test data :", X_test.shape, y_test.shape)

# Preview first few rows
display(X_train.head())

Train data: (4800, 11) (4800,)
Test data : (1200, 11) (1200,)


,age,yrs_experience,family_size,education_level,income,mortgage_amt,credit_card_acct,credit_card_spend,share_trading_acct,fixed_deposit_acct,online_acct
1207,59.0,3.0,4.0,2,111.0,0,0,0.0,0,0,1
1429,49.0,2.0,4.0,3,74.0,220,1,2.5,0,0,1
1347,43.0,1.0,4.0,2,152.0,0,0,0.0,0,0,0
1801,42.0,35.0,1.0,1,87.0,76,1,3.2,0,0,1
4870,57.0,41.0,4.0,3,95.0,105,0,0.0,0,0,0


### 3.2 Optimising Hyperparameters

In [8]:
# 3.2 Optimising Hyperparameters (Model 2 — RandomForest_All Features)

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV

param_grid = [
    {"criterion": ["gini", "entropy"], "n_estimators": range(1, 20)}
]

# Instantiate a grid search object and fit it to the training data
clf = RandomForestClassifier()
grid = GridSearchCV(clf, param_grid, verbose=0, cv=5, scoring='f1')
grid.fit(X_train, y_train)
print(f"Best params: {grid.best_params_}, score: {grid.best_score_:0.2f}")

Best params: {'criterion': 'gini', 'n_estimators': 15}, score: 0.73


### Interpretation of Results  

The **hyperparameter tuning** process successfully identified the optimal parameter settings for both models under the *All Features* configuration.

**Model 1: K-Nearest Neighbours (KNN)**  
- **Best Parameters:** `n_neighbors = 6`, `weights = 'distance'`  
- **Best CV F1-score:** **0.64**  
- **Interpretation:**  
  The optimal number of neighbours (k=6) suggests a balance between **local sensitivity** and **generalisation**.  
  Using **distance-based weighting** allows closer samples to have a stronger influence on predictions, improving recall for the minority class.  
  The moderate F1-score (0.64) indicates reasonable performance but limited non-linear learning capability compared to tree-based models.  

**Model 2: Random Forest (RF)**  
- **Best Parameters:** `criterion = 'gini'`, `n_estimators = 17`  
- **Best CV F1-score:** **0.73**  
- **Interpretation:**  
  The Random Forest achieved a higher F1-score than KNN, showing stronger predictive power on the *All Features* dataset.  
  A small ensemble size (`n_estimators = 17`) was sufficient for stable performance, suggesting low model variance and efficient training.  
  Using the **Gini criterion** indicates that class purity splits contributed effectively to model accuracy.  

**Overall Insight:**  
Random Forest outperformed KNN by approximately **9 percentage points** in F1-score, demonstrating superior ability to capture complex, non-linear relationships among features.  
This result aligns with expectations, as ensemble tree methods are generally more robust to noise and feature scaling than distance-based algorithms.
